### References: https://www.kaggle.com/code/ninamaamary/answering-medqa-questions-with-rag

<div style="background-color: rgba(100, 100, 255, 0.1); padding: 20px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0, 129, 194, 0.5);">
<h1 style="font-size: 26px; font-family: Arial, sans-serif; color: #7e8ff2; text-align: center;"><b>🧬Retrieval-Augmented QA with Gemma2 Meditron-3 </b></h1>
     
<p style="font-size: 18px; font-family: Arial, sans-serif; line-height: 1.6;"> This project focuses on <strong>answering medical exam questions, ensuring that responses are both accurate and grounded by medical textbooks</strong> using <strong>MedQA dataset</strong>. See how retrieval metric, prompt style, top-K and temperature impact accuracy, you’ll discover: 
    <ul style="font-size: 16px; font-family: Arial, sans-serif; list-style-type: square; padding-left: 25px;">
        <li><strong>What you’ll learn:</strong></li>
         <ol>
            <li>How different retrieval methods (BM25 vs. Dot product and Cosine Similarity) impact QA accuracy.</li>
            <li>Experimenting with the different prompting style (zero-shot vs few-shot) RAG pipelines</li>
            <li>The trade-offs when varying context size (top-K retrieved docs) and generation randomness (temperature). </li>
            <li>Which configuration performed best and what that implies for future RAG system design.</li>
        </ol>
        <li><strong>Why you should keep reading:</strong> </li>
        <ol>
            <li>If you’re building domain-specific QA systems, this experiment shows concrete numbers and trends.</li>
            <li>You’ll get a clear visual summary, best-practice takeaways and code you can reuse.  </li>
            <li>For Kaggle readers, this is a ready-to-click, ready-to-run pipeline you can simply fork it.</li>
        </ol>
    </ul>
</div>

<div style="background-color: rgba(200, 100, 255, 0.1); padding: 20px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0, 129, 194, 0.5);">
    <h1 style="font-size: 24px; font-family: Arial, sans-serif; color: #d9534f;"><b>📌 TL;DR (Spoilers)</b></h1>
    
<ul style="font-size: 18px; font-family: Arial, sans-serif;">
        <li>💠 Best configuration on validation dataset:<strong>Dot-Product retrieval + Zero-shot prompting + K=8 + T=0.5</strong>.</li>
        <li>💠 Retrieval metric and prompt style both influence <strong>accuracy meaningfully</strong>.</li>
        <li>💠 Larger context size (higher K) helps <strong>but not always</strong>.</li>
        <li>💠 Temperature tuning remains critical: <strong>too low or too high and performance can degrade</strong>.</li>
    </ul>

<div style="background-color: rgba(100, 100, 255, 0.1); padding: 20px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0, 129, 194, 0.5);">
    <h1 style="font-size: 24px; font-family: Arial, sans-serif; color: #d9534f;"><b>📌 Conclusion and Future Work</b></h1>

<p style="font-size: 18px; font-family: Arial, sans-serif;">
       This experiment shows that even with a cutting-edge model like Gemma2 Meditron-3, <strong>the retrieval and prompting strategy matter more than you’d expect</strong>. Feel free to fork and extend it and Happy Retrieving!  
</p>

<ul style="font-size: 18px; font-family: Arial, sans-serif;">
        <li>✅ The retrieval method <strong>(Dot product vs Cosine Similarity vs BM25) made noticeable differences</strong> suggesting semantic retrieval (Dot product) often edges lexical search (BM25) in this domain. </li>
        <li>✅ <strong>Zero-shot prompting</strong> held its own and even outperformed few-shot in some setups reminding us that “more examples” isn’t always better when context becomes noisy.</li>
        <li>✅ While increasing K did improve accuracy in several runs, gains settled beyond a threshold perhaps due to noise or context window limits.</li>
        <li>✅ <strong>Temperature</strong>, often overlooked, emerged as the “hidden dial” for accuracy <strong>a well-chosen T might boost answers by 10s of percentage points</strong></li>
        <li>✅ The final <strong>29.85% test accuracy</strong>, highlights how challenging out-of-domain reasoning remains, yet also how much can be gained from structured hyperparameter exploration.</li>
    </ul>

<h2 style="font-size: 22px; font-family: Arial, sans-serif; color: #d9534f;"><b>🔍 Future Improvements</b></h2>

<ul style="font-size: 18px; font-family: Arial, sans-serif;">
        <li>🔹 <strong>Enhancing model retrieval:</strong> compare neural-based retrievals such as (Spider, BERT, and Contriever)</li>
        <li>🔹 <strong>Enhancing model generator:</strong> experiment with larger LLMs such as (MedGemma‑4B, MedLlama2‑7B, BioMistral-7B, and BioLlama-7B)</li>
        <li>🔹 <strong>Result interoperability:</strong> incorporate confidence calibration.</li>
    </ul>
</div>

<a id="52"></a>
# 📌 Start From Here if the embeddings have been processed and saved

In [9]:
import re
import os
import glob
import torch
import random
import textwrap
import numpy as np 
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from time import perf_counter as timer
from sentence_transformers import util, SentenceTransformer

KeyboardInterrupt: 

In [ ]:
def load_data(file_path):
    df = pd.read_json(file_path, lines=True)
    return df

In [ ]:
TRAIN_PATH = '../datasets/MedQA-USMLE/questions/US/train.jsonl'
VAL_PATH = '../datasets/MedQA-USMLE/questions/US/dev.jsonl'
TEST_PATH = '../datasets/MedQA-USMLE/questions/US/test.jsonl'

In [ ]:
train_df = load_data(TRAIN_PATH)
val_df = load_data(VAL_PATH)
test_df = load_data(TEST_PATH)

<a id="53"></a>
## Import From File

In [ ]:
embeddings_df_save_path = './text_chunks_and_embeddings.parquet'
text_chunks_and_embeddings_df = pd.read_parquet(embeddings_df_save_path)
text_chunks_and_embeddings_df.head()

<a id="6"></a>
# RAG: Retrieve Using Similarity Search

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Runing on: {device}")

pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")
embeddings = torch.tensor(np.array(text_chunks_and_embeddings_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

In [ ]:
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=device)

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Prepare corpus
corpus = [chunk["chunk_text"] for chunk in pages_and_chunks]
tokenized_corpus = [doc.split(" ") for doc in corpus]

# Initialize BM25 model
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)
    
def extract_specific_answer_letter(output_text, allowed_letters=("A","B","C","D","E","X")):
    text = output_text.strip()

    # 1) Try to find exact 'ANSWER: <letter>' pattern
    m = re.search(r"ANSWER\s*[:\-]\s*([A-EX])\b", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # 2) Try 'REASONING:' followed by newline(s) then 'ANSWER:' etc.
    m = re.search(r"REASONING:.*?ANSWER\s*[:\-]\s*([A-EX])\b", text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).upper()

    # 3) Otherwise search for the first standalone capital letter among allowed letters
    m = re.search(r"\b([A-EX])\b", text)
    if m:
        return m.group(1).upper()

    # 4) Last resort: look for single letter at start or end
    m = re.match(r"^([A-EX])\b", text)
    if m:
        return m.group(1).upper()

    m = re.search(r"\b([A-EX])\.?$", text)
    if m:
        return m.group(1).upper()

    # fallback
    return "X"

 
def retrieve_relevant_resources(query, embeddings, model=embedding_model, n_resources_to_return=10, metric='dot_product', print_time=True, show_progress_bar=False):
    start_time = timer()

    if metric.lower() == "bm25":
        tokenized_query = query.split(" ")
        scores = bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_resources_to_return]
        top_scores = [scores[i] for i in top_indices]
        metric_name = "BM25"
    
    elif metric.lower() in ["dot_product", "cosine_similarity"]:
        query_embedding = model.encode(query, convert_to_tensor=True, show_progress_bar=show_progress_bar)

        if metric.lower() == 'cosine_similarity':
            scores = util.cos_sim(query_embedding, embeddings)[0]
            metric_name = 'Cosine Similarity'

        elif metric.lower() == 'dot_product':
            scores = util.dot_score(query_embedding, embeddings)[0]
            metric_name = 'Dot Product'

        top_scores, top_indices = torch.topk(input=scores, k=n_resources_to_return)
        top_scores = top_scores.tolist()
        top_indices = top_indices.tolist()

    end_time = timer()

    if print_time:
        print(f"Time taken to get {metric_name} scores: {end_time - start_time:.5f} seconds.")

    return top_scores, top_indices
       
def print_top_results_and_scores(query, embeddings, pages_and_chunks=pages_and_chunks, n_resources_to_return=2, metric='dot_product'):
    scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings, n_resources_to_return=n_resources_to_return, metric=metric)
    
    print(f"Query: {query}\n")
    print("Results:")

    for score, index in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print_wrapped(pages_and_chunks[index]["chunk_text"])
        print("\n")

<a id="61"></a>
## Using Dot Product 

In [ ]:
query = "Weight loss"
print_top_results_and_scores(query, embeddings, metric='dot_product')

<a id="62"></a>
## Using Cosine Similarity

In [ ]:
query = "Weight loss"
print_top_results_and_scores(query, embeddings, metric='cosine_similarity')

Most modern Sentence Transformer models (like the all-mpnet-base-v2 you are using in the example) are designed to produce normalized embeddings by default. When a vector is normalized, its length ($\|\vec{A}\|$) is 1, therefore, Dot Product and Cosine Similarity are mathematically identical.$$\text{Cosine Similarity} = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|} = \frac{\vec{A} \cdot \vec{B}}{1 \cdot 1} = \vec{A} \cdot \vec{B} = \text{Dot Product}$$

<a id="7"></a>
# RAG: Generating with Local LLM

**⚠️You need to create a hugging face account, give permission for `google/gemma-2b-it` and then create access token and use it to login. however we are using `OpenMeditron/Meditron3-Gemma2-2B` which does not require access token**. if you are changing the LMM please uncomment the code block below

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_llm_model(model_id, device="cuda"):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="sdpa",
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model.eval()
    
    print(f"Model {model_id} loaded successfully on {device}.")
    return model, tokenizer

In [ ]:
gemma_id = "OpenMeditron/Meditron3-Gemma2-2B" 
gemma_model, gemma_model_tokenizer = load_llm_model(gemma_id, device)

<a id="71"></a>
## LLM Model Stats

In [ ]:
def get_model_mem_size(model, model_id):   
    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])

    model_mem_bytes = mem_params + mem_buffers 
    model_mem_mb = model_mem_bytes / (1024**2)
    model_mem_gb = model_mem_bytes / (1024**3)

    stats = {
        'model_mem_bytes': model_mem_bytes,
        'model_mem_mb': round(model_mem_mb, 2),
        'model_mem_gb': round(model_mem_gb, 2)
    }
    print(f"Model stats of: {model_id}")
    display(pd.Series(stats, name='Model Information').to_frame())

In [ ]:
get_model_mem_size(gemma_model, gemma_id)

<a id="8"></a>
# RAG: Augmentation + Generation

In [ ]:
import gc

def zero_shot_prompt_formatter(question, options, context_items, tokenizer):
    # Create a clean, readable string from the retrieved context
    context = "\n".join([f"Reference {i+1}: {item['chunk_text']}" for i, item in enumerate(context_items)])
    option_str = "\n".join([f"{key}: {value}" for key, value in options.items()])

    base_prompt = f"""You are a medical expert. Use ONLY the reference material below to answer the exam question.

REFERENCE MATERIAL:
{context}

EXAM QUESTION:
{question}

OPTIONS:
{option_str}

TASK:
Identify the single best answer supported by the reference material.

REQUIRED FORMAT (exactly):
ANSWER: <single letter: A, B, C, D or E ; if none is supported, answer X>

Important: Do NOT add any other text, punctuation, or labels. Do not use external knowledge; rely only on the reference material above.
"""
    
    dialogue_template = [{"role": "user", "content": base_prompt}]
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template, tokenize=False, add_generation_prompt=True)
    return prompt


def few_shot_prompt_formatter(question, options, context_items, tokenizer, max_example_chars=700):
    def trim(text, n=max_example_chars):
        return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + " ..."

    context = "\n".join([f"Reference {i+1}: {trim(item['chunk_text'])}" for i, item in enumerate(context_items)])
    option_str = "\n".join([f"{k}: {v}" for k, v in options.items()])

    example_1 = (
        "REFERENCE MATERIAL:\n"
        "Streptococcal pharyngitis typically presents with fever, sore throat,"
        "tender cervical nodes, and tonsillar exudates. Viral pharyngitis more commonly,"
        "causes cough and rhinorrhea.\n"
        "EXAM QUESTION:\n"
        "A patient has sore throat, fever, and tender cervical nodes but no cough. Which is most likely?\n\n"
        "OPTIONS:\nA: Viral pharyngitis\nB: Streptococcal pharyngitis\nC: Allergic rhinitis\nD: Infectious mononucleosis\nE: Cancer\n"
        "ANSWER: B"
    )
    base_prompt = f"""You are a medical expert. Use ONLY the reference material below to answer the exam question.

EXAMPLES (follow the exact format):

{example_1}

---  # Now the real example to answer

REFERENCE MATERIAL:
{context}

EXAM QUESTION:
{question}

OPTIONS:
{option_str}

TASK:
Identify the single best answer supported by the reference material.

REQUIRED FORMAT (exactly):
ANSWER: <single letter: A, B, C, D, E ; if none is supported, answer X>

Important: Do NOT add any other text, punctuation, or labels. Do not use external knowledge; rely only on the reference material above.
"""
    dialogue_template = [{"role": "user", "content": base_prompt}]
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template, tokenize=False, add_generation_prompt=True)
    return prompt
    
def ask(question, options, temperature=0.3, max_new_tokens=512, format_answer_text=True, return_answer_only=True, num_retrieved_documents=5, metric="dot_product", prompt_type="zero_shot", llm_model=gemma_model, top_p=0.9, repetition_penalty=1.0, tokenizer=gemma_model_tokenizer):
    # Retrieval
    scores, indices = retrieve_relevant_resources(
        query=question, 
        embeddings=embeddings, 
        n_resources_to_return=num_retrieved_documents, 
        metric=metric, 
        print_time=False
    )
    
    context_items = [pages_and_chunks[i] for i in indices[:num_retrieved_documents]]
    
    for i, item in enumerate(context_items):
        item["score"] = float(scores[i].cpu().item() if hasattr(scores[i], "cpu") else scores[i])

    if prompt_type == "few_shot":
        prompt = few_shot_prompt_formatter(question=question, options=options, context_items=context_items, tokenizer=tokenizer)
    elif prompt_type == "zero_shot":
        prompt = zero_shot_prompt_formatter(question=question, options=options, context_items=context_items, tokenizer=tokenizer)

    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **input_ids, 
            temperature=temperature, 
            do_sample=True, 
            max_new_tokens=max_new_tokens, 
            top_p=top_p, 
            repetition_penalty=repetition_penalty
        )

    output_text = tokenizer.decode(outputs[0])

    # Post-processing
    if format_answer_text:
        # Simple cleanup of common LLM artifacts
        output_text = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "").strip()
        output_text = re.sub(r'Sure, here is the answer to the user query:\s*\n*', '', output_text)

    if return_answer_only:
        return output_text

    del input_ids, outputs  
    torch.cuda.empty_cache() 
    gc.collect()
    
    return output_text, context_items

<a id="81"></a>
## Look at RAG-Pipeline 

In [ ]:
import torch._inductor.config as inductor_config
import warnings

warnings.filterwarnings("ignore", message="Not enough SMs to use max_autotune_gemm mode")

In [ ]:
def display_options(options):
    print("Options:")
    for key, value in options.items():
        print(f"  {key}: {value}")

<a id="82"></a>
## Get Random Question

In [ ]:
random_sample = train_df.sample(n=1)
question = random_sample.iloc[0]["question"]
options = random_sample.iloc[0]["options"]
ground_truth = random_sample.iloc[0]["answer"]
ans_indx = random_sample.iloc[0]["answer_idx"]

<a id="83"></a>
## Using Zero-Shot Learning

In [ ]:
print(f"Question: {question}")
display_options(options)

answer, context_items = ask(question=question, options=options, temperature=0.3, max_new_tokens=512, return_answer_only=False, llm_model=gemma_model, tokenizer=gemma_model_tokenizer)

print(f"RAG Answer:\n")
print_wrapped(answer)
print(f"Actual Answer: \n {ans_indx}: {ground_truth}")

In [ ]:
extract_specific_answer_letter(answer)

<a id="84"></a>
## Using Few-Shot Learning

In [ ]:
print(f"Question: {question}")
display_options(options)

answer, context_items = ask(question=question, options=options, temperature=0.3, max_new_tokens=512, prompt_type="few_shot", return_answer_only=False, llm_model=gemma_model, tokenizer=gemma_model_tokenizer)

print(f"RAG Answer:\n")
print_wrapped(answer)
print(f"Actual Answer: \n {ans_indx}: {ground_truth}")

In [ ]:
extract_specific_answer_letter(answer)

<a id="9"></a>
# Hyperparameter Tuning (Using VAL Dataset)

In [ ]:
def run_validation_experiment(metrics, prompt_types, top_k_values, temperatures, llm_configs, validation_dataset, top_p_values, repetition_penalties, max_questions=None):
    results = defaultdict(lambda: {
        'qa_correct': 0, 'total': 0, 'qa_accuracy': 0.0,
    })

    validation_dataset = validation_dataset.to_dict('records')

    dataset_size = len(validation_dataset)
    
    random.seed(42) 
    sampled_dataset = random.sample(validation_dataset, max_questions) 
    print(f"Sampling {max_questions} questions from a total of {dataset_size} for hyper-tuning.")

    sampled_size = len(sampled_dataset)

    configurations = []
    for llm_config in llm_configs:
        llm_name = llm_config['name']
        for metric in metrics:
            for p_type in prompt_types:
                for k in top_k_values:
                    for temp in temperatures:
                        for top_p in top_p_values:
                            for rep_penalty in repetition_penalties:
                                configurations.append((llm_config, metric, p_type, k, temp, top_p, rep_penalty))
    
    print(f"Running {len(configurations) * sampled_size} LLM inferences across {len(configurations)} configurations (using {sampled_size} questions)")

    for llm_config, metric, p_type, k, temp, top_p, rep_penalty in configurations:
        llm_name = llm_config['name']
        current_llm_model = llm_config['model']       
        current_tokenizer = llm_config['tokenizer']
        
        config_key = f"{llm_name}_{metric}_P{p_type}_K{k}_T{temp}_TP{top_p}_RP{rep_penalty}"
        print(f"\nTesting: {config_key}")
        
        qa_correct_count = 0
        total_count = 0
        
        for item in tqdm(sampled_dataset, desc=f"Evaluating {config_key}"):
            question = item['question']
            options = item['options']
            true_answer_letter = item['answer_idx'] 
            
            scores, indices = retrieve_relevant_resources(
                query=question,  
                embeddings=embeddings,
                n_resources_to_return=k, 
                metric=metric, 
                print_time=False
            )
            
            llm_output = ask(
                question=question,
                options=options,
                num_retrieved_documents=k,
                metric=metric,
                prompt_type=p_type,
                llm_model=current_llm_model,
                tokenizer=current_tokenizer,
                temperature=temp,
                top_p=top_p,
                repetition_penalty=rep_penalty,
                return_answer_only=True
            )
            
            predicted_letter = extract_specific_answer_letter(llm_output)
            
            if str(predicted_letter).upper() == str(true_answer_letter).upper():
                qa_correct_count += 1
            
            total_count += 1
            torch.cuda.empty_cache()
            gc.collect()
            
        qa_accuracy = (qa_correct_count / total_count) * 100 if total_count > 0 else 0
        
        results[config_key] = {
            'qa_correct': qa_correct_count,
            'total': total_count,
            'qa_accuracy': qa_accuracy,
        }
        
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"{config_key} QA Accuracy: {qa_accuracy:.2f}%")

    print("\nExperiment Summary")
    for key, res in results.items():
        print(f"{key}: QA Acc={res['qa_accuracy']:.2f}%")
        
    return results

In [ ]:
metrics_to_test = ["dot_product"]
prompt_types_to_test = ["zero_shot", "few_shot"]
top_k_values_to_test = [5, 8] 
temperatures_to_test = [0.1, 0.2]
top_p_values_to_test = [0.85, 0.9]
repetition_penalties_to_test = [1.1, 1.15]

llm_configs = [
    {"name": "Gemma2 Meditron-3", "model": gemma_model, "tokenizer": gemma_model_tokenizer},
]


final_results = run_validation_experiment(
    metrics=metrics_to_test,
    prompt_types=prompt_types_to_test,
    top_k_values=top_k_values_to_test,
    llm_configs=llm_configs, 
    validation_dataset=val_df,
    temperatures= temperatures_to_test,
    repetition_penalties=repetition_penalties_to_test,
    top_p_values=top_p_values_to_test,
    max_questions=100
)

<a id="91"></a>
## Plotting the Experiments

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import re
import textwrap

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "axes.facecolor": "#f9f9f9",
    "figure.facecolor": "#ffffff",
    "axes.edgecolor": "#d0d0d0",
    "grid.color": "#e5e5e5",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "axes.spines.right": False,
    "axes.spines.top": False,
})
cl = "coolwarm"

def save_to_csv(results_dict):
    df = clean_results_to_df(results_dict)
    df.to_csv('results.csv')
    return df 
    
def clean_results_to_df(results_dict):
    records = []
    for key, res in results_dict.items():
        match = re.search(
            r"(.+?)_(dot_product|bm25)_P(zero_shot|few_shot)_K(\d+)_T([\d.]+)_TP([\d.]+)_RP([\d.]+)",
            key,
            re.IGNORECASE
        )
        if not match:
            continue
        llm_name, metric, prompt_type, k, temp, tp, rp = match.groups()
        records.append({
            "llm_name": llm_name,
            "metric": metric.lower(),
            "prompt_type": prompt_type.lower(),
            "top_k": int(k),
            "temperature": float(temp),
            "top_p":tp,
            "repetition_penalty":rp,
            "qa_accuracy": res["qa_accuracy"],
        })

    
    return pd.DataFrame(records)


def plot_line_subplots(df, x_feature, fixed_feature, title, xlabel, ylabel="Accuracy (%)"):
    categories = sorted(df[fixed_feature].unique())
    fig, axes = plt.subplots(1, len(categories), figsize=(5 * len(categories), 5), sharey=True)
    if len(categories) == 1:
        axes = [axes]

    palette = sns.color_palette(cl, n_colors=df["metric"].nunique() if fixed_feature == "prompt_type" else df["prompt_type"].nunique())

    for ax, cat in zip(axes, categories):
        subset = df[df[fixed_feature] == cat].sort_values(by=x_feature)
        sns.lineplot(
            data=subset,
            x=x_feature,
            y="qa_accuracy",
            hue="metric" if fixed_feature == "prompt_type" else "prompt_type",
            marker="o",
            linewidth=2.5,
            ax=ax,
            palette=palette
        )
        ax.set_title(f"{fixed_feature.capitalize()}: {cat}", pad=15)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(frameon=False, title="", loc="best")

    fig.suptitle(title, fontsize=16, weight="bold", y=1.03)
    plt.tight_layout()
    plt.show()


def plot_bar_subplots(df, feature, title, xlabel, ylabel="Accuracy (%)"):
    categories = df[feature].unique()
    n = len(categories)
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 6), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, cat in zip(axes, categories):
        subset = df[df[feature] == cat].copy()
        subset["config"] = subset.apply(
            lambda r: f"{r['prompt_type']} | K={r['top_k']} | T={r['temperature']}", axis=1
        )
        subset = subset.sort_values("qa_accuracy", ascending=False).head(6)

        sns.barplot(
            data=subset,
            x="config",
            y="qa_accuracy",
            hue="temperature",
            palette=cl,
            ax=ax,
            edgecolor="none",
            errorbar=None
        )

        ax.set_title(f"{feature.capitalize()}: {cat}", fontsize=13, pad=15)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel if ax == axes[0] else "")
        ax.set_xticklabels(
            [textwrap.fill(l, 15) for l in subset["config"].unique()],
            rotation=30, ha="right", fontsize=9
        )
        ax.legend(title="Temp", fontsize=9, frameon=False, loc="best")
        ax.set_ylim(0, df["qa_accuracy"].max() * 1.2)
        for c in ax.containers:
            ax.bar_label(c, fmt="%.1f%%", fontsize=8, padding=2)

    fig.suptitle(title, fontsize=15, weight="bold", y=1.03)
    plt.tight_layout()
    plt.show()


def plot_combined_bar(df, title="Accuracy by Metric and Prompt Type"):
    plt.figure(figsize=(8, 6))
    grouped = (
        df.groupby(["metric", "prompt_type"])["qa_accuracy"]
        .mean()
        .reset_index()
    )

    sns.barplot(
        data=grouped,
        x="metric",
        y="qa_accuracy",
        hue="prompt_type",
        palette=cl,
        edgecolor="white",
        linewidth=0.8,
        errorbar=None
    )

    plt.title(title, fontsize=15, weight="bold", pad=15)
    plt.xlabel("Retrieval Metric", fontsize=12)
    plt.ylabel("Accuracy (%)", fontsize=12)
    plt.ylim(0, df["qa_accuracy"].max() * 1.1)
    plt.legend(title="Prompt Type", frameon=False, fontsize=10, loc="best")
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()


def plot_hyperparam_results(results_dict):
    df = clean_results_to_df(results_dict)
    plot_line_subplots(
        df,
        x_feature="temperature",
        fixed_feature="prompt_type",
        title="Accuracy vs Temperature (by Prompt Type)",
        xlabel="Temperature"
    )

    # plot_line_subplots(
    #     df,
    #     x_feature="top_k",
    #     fixed_feature="metric",
    #     title="Accuracy vs Top-K (by Retrieval Metric)",
    #     xlabel="Top-K"
    # )

    plot_combined_bar(
        df,
        title="Accuracy Comparison by Retrieval Metric and Prompt Type"
    )

    plot_bar_subplots(
        df,
        feature="prompt_type",
        title="Top 6 Configurations per Promp type",
        xlabel="Config (Prompt / K / Temp)"
    )

    plot_bar_subplots(
        df,
        feature="metric",
        title="Top 6 Configurations per Metric",
        xlabel="Config (Prompt / K / Temp)"
    )

    return df

In [ ]:
save_to_csv(final_results)

In [ ]:
results_df = plot_hyperparam_results(final_results)

In [ ]:
best_config = max(final_results.items(), key=lambda kv: kv[1]['qa_accuracy'])
print("Best configuration:", best_config[0], best_config[1])

<a id="10"></a>
# Evaluation Pipeline

Now we can use the parameters from our hypertuning from the validation dataset and evaluate performance on the test set

In [ ]:
import re

def evaluate_rag_model(df_test, best_config_key, llm_configs, embeddings):
    match = re.search(r"(.+?)_(dot_product|bm25)_P(zero_shot|few_shot)_K(\d+)_T([\d.]+)_TP([\d.]+)_RP([\d.]+)", best_config_key, re.IGNORECASE)

    llm_name, metric, prompt_type, k, temp, top_p, rep_penalty = match.groups()
    metric = metric.lower()
    prompt_type = prompt_type.lower()
    k = int(k)
    temp = float(temp)
    top_p = float(top_p)
    rep_penalty = float(rep_penalty)

    print(f"\nEvaluating using best config| LLM:{llm_name}, metric:{metric}, prompt_type:{prompt_type}, top_k:{k}, temp:{temp}, top_:{top_p}, rep_penalty:{rep_penalty}\n")

    # Get LLM model & tokenizer
    llm_entry = next((c for c in llm_configs if c['name'].lower() == llm_name.lower()), None)

    model = llm_entry['model']
    tokenizer = llm_entry['tokenizer']

    results_list = []

    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"Evaluating {llm_name}", position=0, leave=True):
        question = row['question']
        options = row['options']
        true_answer_letter = str(row['answer_idx']).strip().upper()

        llm_response = ask(
            question=question,
            options=options,
            num_retrieved_documents=k,
            metric=metric,
            prompt_type=prompt_type,
            llm_model=model,
            tokenizer=tokenizer,
            temperature=temp,
            max_new_tokens=512,
            top_p=top_p,
            repetition_penalty=rep_penalty,
            return_answer_only=True
        )

        predicted_letter = extract_specific_answer_letter(llm_response)
        is_correct = (predicted_letter.upper() == true_answer_letter.upper())

        results_list.append({
            'question': question,
            'options': options,
            'true_answer': true_answer_letter,
            'predicted_letter': predicted_letter,
            'is_correct': is_correct,
            'llm_response': llm_response
        })

    results_df = pd.DataFrame(results_list)
    acc = results_df['is_correct'].mean() * 100
    print(f"\nFinal Evaluation Accuracy ({best_config_key}): {acc:.2f}%")
    return results_df

In [ ]:
# 1. Extract best config
best_config_key, best_config_stats = max(final_results.items(), key=lambda kv: kv[1]['qa_accuracy'])
print("Best config:", best_config_key, best_config_stats)

# 2. Evaluate test set with it
test_results = evaluate_rag_model(
    df_test=test_df,
    best_config_key=best_config_key,
    llm_configs=llm_configs,
    embeddings=embeddings
)

<a id="101"></a>
## Checking Evaluation Results

In [ ]:
empty_count = (test_results['predicted_letter'] == '').sum()
total_samples = len(test_results)

print(f"Total samples evaluated: {total_samples}")
print(f"Count of empty predicted letters: {empty_count}")

if total_samples > 0:
    empty_percentage = (empty_count / total_samples) * 100
    print(f"Percentage of un-extractable answers: {empty_percentage:.2f}%")

In [ ]:
unextracted_answers_df = test_results[~test_results['predicted_letter'].isin(['A', 'B', 'C', 'D', 'E'])]

print(f"Total un-extractable answers (not A, B, C, D, or E): {len(unextracted_answers_df)}")

if not unextracted_answers_df.empty:
    print("\nFirst 5 failed extractions:")
    display(unextracted_answers_df[['ground_truth', 'llm_response', 'predicted_letter']].head(5))